# Aiscern — Audio Deepfake Detector Fine-tuning
**Model**: `facebook/wav2vec2-base` + LoRA (trains in ~2h on Kaggle P100)

**Run on**: Kaggle (30h free GPU/week) → https://www.kaggle.com/code

**Output**: `saghi776/aiscern-audio-detector` on HuggingFace

**Accuracy target**: 94%+ (up from current 91%)

In [ ]:
# ── STEP 1: Install deps ─────────────────────────────────────────────────────
!pip install -q transformers datasets peft accelerate soundfile librosa \
             evaluate scikit-learn huggingface_hub

In [ ]:
# ── STEP 2: Config ───────────────────────────────────────────────────────────
import os

HF_TOKEN        = 'YOUR_HF_TOKEN_HERE'  # your token
BASE_MODEL      = 'facebook/wav2vec2-base'
PUSH_TO_REPO    = 'saghi776/aiscern-audio-detector'
SOURCE_DATASET  = 'saghi776/detectai-dataset'

# Training config
MAX_SAMPLES     = 8000   # per class — fits in Kaggle P100 RAM
MAX_DURATION_S  = 5      # clip audio to 5 seconds
SAMPLE_RATE     = 16000
BATCH_SIZE      = 16
EPOCHS          = 5
LR              = 2e-4

os.environ['HF_TOKEN'] = HF_TOKEN
print('Config set ✅')

In [ ]:
# ── STEP 3: Load data from best audio datasets ──────────────────────────────
from datasets import load_dataset, concatenate_datasets, Audio, Dataset
import numpy as np

print('Loading ASVspoof2019 (gold standard deepfake audio dataset)...')
ds_spoof = load_dataset(
    'DynamicSuperb/AudioDeepfakeDetection_ASVspoof2019LA',
    split='train', token=HF_TOKEN
).cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))

# Map labels to 0=real, 1=fake
label_map = {'bonafide': 0, 'genuine': 0, 'spoof': 1, 'spoofed': 1}
def map_label(batch):
    batch['label'] = label_map.get(str(batch['label']).lower(), -1)
    return batch

ds_spoof = ds_spoof.map(map_label)
ds_spoof = ds_spoof.filter(lambda x: x['label'] != -1)

print('Loading WaveFake (GAN vocoder fakes)...')
ds_wavefake = load_dataset('balt0/WaveFake', split='train', token=HF_TOKEN)\
    .cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))

print('Loading in-the-wild deepfakes...')
ds_wild = load_dataset(
    'motheecreator/in-the-wild-audio-deepfake', 
    split='train', token=HF_TOKEN
).cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))

wild_label_map = {'spoof': 1, 'fake': 1, 'bonafide': 0, 'real': 0}
ds_wild = ds_wild.map(map_label)
ds_wild = ds_wild.filter(lambda x: x['label'] != -1)

# WaveFake = all fake (label 1)
ds_wavefake = ds_wavefake.map(lambda x: {'label': 1})

# Combine and balance
combined = concatenate_datasets([
    ds_spoof.select_columns(['audio','label']),
    ds_wavefake.select_columns(['audio','label']),
    ds_wild.select_columns(['audio','label']),
])

real = combined.filter(lambda x: x['label'] == 0).shuffle(42).select(range(min(MAX_SAMPLES, len(combined.filter(lambda x: x['label']==0)))))
fake = combined.filter(lambda x: x['label'] == 1).shuffle(42).select(range(min(MAX_SAMPLES, len(combined.filter(lambda x: x['label']==1)))))
n = min(len(real), len(fake))
balanced = concatenate_datasets([real.select(range(n)), fake.select(range(n))]).shuffle(42)

split = balanced.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'Train: {len(train_ds):,}  Eval: {len(eval_ds):,}')

In [ ]:
# ── STEP 4: Feature extraction ───────────────────────────────────────────────
from transformers import Wav2Vec2FeatureExtractor

extractor = Wav2Vec2FeatureExtractor.from_pretrained(BASE_MODEL)

def preprocess(batch):
    arrays = []
    for a in batch['audio']:
        arr = a['array']
        # Clip to MAX_DURATION_S
        arr = arr[:SAMPLE_RATE * MAX_DURATION_S]
        # Pad if too short
        if len(arr) < SAMPLE_RATE:
            arr = np.pad(arr, (0, SAMPLE_RATE - len(arr)))
        arrays.append(arr.astype(np.float32))
    inputs = extractor(arrays, sampling_rate=SAMPLE_RATE, 
                       return_tensors='np', padding=True, 
                       max_length=SAMPLE_RATE * MAX_DURATION_S, 
                       truncation=True)
    batch['input_values']     = inputs.input_values
    batch['attention_mask']   = inputs.get('attention_mask', 
                                 np.ones_like(inputs.input_values))
    return batch

train_ds = train_ds.map(preprocess, batched=True, batch_size=32, 
                         remove_columns=['audio'])
eval_ds  = eval_ds.map(preprocess,  batched=True, batch_size=32,
                         remove_columns=['audio'])
train_ds.set_format('torch')
eval_ds.set_format('torch')
print('Features extracted ✅')

In [ ]:
# ── STEP 5: Load model + apply LoRA ─────────────────────────────────────────
from transformers import Wav2Vec2ForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
import torch

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'real', 1: 'fake'},
    label2id={'real': 0, 'fake': 1},
    ignore_mismatched_sizes=True,
)

# Freeze feature extractor — only fine-tune transformer layers
model.freeze_feature_extractor()

# LoRA config — trains only 0.5% of parameters
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,              # rank
    lora_alpha=32,
    lora_dropout=0.1,
    bias='none',
    target_modules=['q_proj','v_proj','k_proj','out_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'Model on {device} ✅')

In [ ]:
# ── STEP 6: Train ────────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load('accuracy')
f1       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'f1':       f1.compute(predictions=preds, references=labels, average='binary')['f1'],
    }

training_args = TrainingArguments(
    output_dir='./audio-detector-checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    push_to_hub=True,
    hub_model_id=PUSH_TO_REPO,
    hub_token=HF_TOKEN,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to='none',
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print('Training complete ✅')

In [ ]:
# ── STEP 7: Evaluate + Push to HF ───────────────────────────────────────────
results = trainer.evaluate()
print(f"\nFinal Results:")
print(f"  Accuracy: {results['eval_accuracy']:.4f}")
print(f"  F1 Score: {results['eval_f1']:.4f}")

# Push model + extractor to HuggingFace
trainer.push_to_hub(commit_message='Fine-tuned wav2vec2 audio deepfake detector')
extractor.push_to_hub(PUSH_TO_REPO, token=HF_TOKEN)

print(f'\n✅ Model pushed to: https://huggingface.co/{PUSH_TO_REPO}')